# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

My baseline rule is designed to prioritize content pages that may deserve review for refresh opportunities.

The rule assigns a higher score to pages that:

- Have high search visibility (more impressions)
- Are older and potentially stale
- Have relatively low CTR compared with other pages

The goal is not to predict recovery or prove causation. The goal is to create a transparent ranking that helps a reviewer decide which pages to inspect first.

Pages with the highest scores are considered stronger refresh candidates because they combine meaningful visibility with signs of age or underperformance.

## Reason Codes

### stale_visible_page

The page is relatively old and still receives meaningful visibility. This suggests there may be value in reviewing or refreshing the content.

### monitor

The page does not currently meet the baseline criteria for refresh review. It should continue to be monitored but is not a priority candidate.

## Action Labels

### REFRESH

Recommended when a page receives the reason code `stale_visible_page`.

### MONITOR

Recommended when a page receives the reason code `monitor`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

The baseline score combines three observable signals:

- Visibility (impressions)
- Content age (staleness)
- CTR opportunity

Higher scores indicate pages that may deserve review before lower-scoring pages.

The output is a ranked queue containing:

- content_id
- baseline_score
- reason_code
- action_label

In [2]:
import os

os.makedirs("../../work/outputs", exist_ok=True)

In [3]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Baseline scoring rule
df["baseline_score"] = (
    0.40 * (df["impressions_90d"] / df["impressions_90d"].max())
    +
    0.30 * (df["content_age_days"] / df["content_age_days"].max())
    +
    0.30 * (
        1 - (
            df["ctr"] /
            max(df["ctr"].max(), 0.01)
        )
    )
) * 100

# Reason code
df["reason_code"] = np.where(
    (df["content_age_days"] >= 180)
    &
    (df["impressions_90d"] >= 500),
    "stale_visible_page",
    "monitor"
)

# Action label
df["action_label"] = np.where(
    df["reason_code"] == "stale_visible_page",
    "REFRESH",
    "MONITOR"
)

# Rank pages
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

# Save CSV
queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(
    "../../work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV created successfully")

# Display top 10
queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(10)

CSV created successfully


,content_id,baseline_score,reason_code,action_label
6653,content_5fe46e04994d,98.521830,stale_visible_page,REFRESH
17812,content_aaef01a50def,93.548392,stale_visible_page,REFRESH
26844,content_8c19996aa890,92.971339,stale_visible_page,REFRESH
21819,content_4c36c775b818,89.327748,stale_visible_page,REFRESH
29879,content_1a9e894be2e2,87.724441,stale_visible_page,REFRESH
18870,content_db5989a78dd3,80.271381,stale_visible_page,REFRESH
29400,content_2dba2b1f9536,80.102113,stale_visible_page,REFRESH
21565,content_9532f197bbc8,77.298186,stale_visible_page,REFRESH
19636,content_2cb567c3c89b,76.563973,monitor,MONITOR
13537,content_2c2606c5d176,75.937264,stale_visible_page,REFRESH


In [4]:
import os

os.path.exists(
    "../../work/outputs/baseline_action_score.csv"
)

True

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue.head(20)

top20[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
]

,content_id,baseline_score,reason_code,action_label
6653,content_5fe46e04994d,98.521830,stale_visible_page,REFRESH
17812,content_aaef01a50def,93.548392,stale_visible_page,REFRESH
26844,content_8c19996aa890,92.971339,stale_visible_page,REFRESH
21819,content_4c36c775b818,89.327748,stale_visible_page,REFRESH
29879,content_1a9e894be2e2,87.724441,stale_visible_page,REFRESH
18870,content_db5989a78dd3,80.271381,stale_visible_page,REFRESH
29400,content_2dba2b1f9536,80.102113,stale_visible_page,REFRESH
21565,content_9532f197bbc8,77.298186,stale_visible_page,REFRESH
19636,content_2cb567c3c89b,76.563973,monitor,MONITOR
13537,content_2c2606c5d176,75.937264,stale_visible_page,REFRESH


# Top-20 Review

### 1.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may be experiencing normal seasonality rather than a true content decline.

---

### 2.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Another related page may have absorbed traffic from this page.

---

### 3.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may already have been updated recently and the changes are not yet reflected in the data.

---

### 4.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Traffic loss may be temporary rather than persistent.

---

### 5.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Demand for the topic may have genuinely declined.

---

### 6.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may already perform well enough despite its age.

---

### 7.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
A site-wide traffic decline could be affecting the page.

---

### 8.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Position changes outside content quality may explain the performance shift.

---

### 9.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Measurement noise may make the opportunity appear larger than it is.

---

### 10.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may already satisfy user intent effectively.

---

### 11.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Recent industry trends may have reduced demand.

---

### 12.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Observed signals may reflect a temporary fluctuation.

---

### 13.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may have limited improvement potential.

---

### 14.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Competitive changes may be driving performance differences.

---

### 15.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The topic may naturally receive lower engagement.

---

### 16.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may already be protected by strong rankings.

---

### 17.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page could recover without intervention.

---

### 18.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The opportunity may be overstated due to limited data.

---

### 19.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
Changes in search behavior may explain the observed signals.

---

### 20.
Action: REFRESH

Reason Code: stale_visible_page

Confidence: Medium

What would make it wrong?
The page may already be meeting business goals despite lower metrics.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak Picks + Leakage Check

## Weak Picks

Some recommendations may be weak despite receiving a relatively high baseline score.

### Potential Weak Pick 1

Why it may be wrong:

The page is old and visible, but the observed performance may be driven by normal seasonality rather than a genuine content issue.

### Potential Weak Pick 2

Why it may be wrong:

Traffic may have shifted to a related page, making this page appear weaker than it actually is.

### Potential Weak Pick 3

Why it may be wrong:

The page may have been updated recently and the data may not yet reflect those changes.

### Potential Weak Pick 4

Why it may be wrong:

The score is driven primarily by age rather than multiple independent signals.

### Potential Weak Pick 5

Why it may be wrong:

The decline may represent temporary noise rather than a persistent pattern.

## Leakage Check

I reviewed the inputs used in the baseline rule.

Included features:

- impressions_90d
- content_age_days
- ctr

Excluded features:

- trend_direction
- is_declining_label
- any future-window metrics
- any product decision flags
- any derived target fields

### Product Flag Check

No FlyRank product outputs such as health scores, priority scores, action types, refresh flags, or decision flags were used in the rule.

### Future Window Check

The rule uses only observable signals available before the review decision.

No future performance measurements were included.

### Conclusion

The baseline score is intended to be leakage-safe. It relies only on historical visibility, CTR, and content age signals that would be available at the time a reviewer receives the recommendation.

In [7]:
queue.tail(5)[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
]

,content_id,baseline_score,reason_code,action_label
22607,content_bc2c0c7243df,15.638375,monitor,MONITOR
13661,content_bf398aa7400e,15.638375,monitor,MONITOR
19341,content_4272d3a330a3,7.659652,monitor,MONITOR
240,content_006b16e7a2e7,7.446886,monitor,MONITOR
6473,content_cfa4d9f1bf0a,5.957524,monitor,MONITOR


## Self-check

Before you submit, confirm each line honestly:

- [ y] Every section above is filled — markdown thinking AND the code that backs it
- [ y] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ y] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ y] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.